In [4]:
# ---------------------------------------------------------
# GOLD LAYER SETUP
# Purpose: Load Silver tables and sample a consistent subset
# of listings for fast exploration and feature engineering.
# Sampling ensures that calendar, reviews, and listings are
# all aligned to the same set of listing_ids.
# ---------------------------------------------------------

import pandas as pd
from pathlib import Path

# Resolve project root (assumes notebook is in /notebooks)
notebook_dir = Path().resolve()
project_root = notebook_dir.parent
silver_dir = project_root / "data" / "silver"

# Load Silver tables
df_listings = pd.read_parquet(silver_dir / "listings_clean.parquet")
df_calendar = pd.read_parquet(silver_dir / "calendar_clean.parquet")
df_reviews = pd.read_parquet(silver_dir / "reviews_clean.parquet")

In [ ]:
# # ---------------------------------------------------------
# # UNCOMMENT WHEN EXPORING/PREPPING
# # ---------------------------------------------------------

# # ---------------------------------------------------------
# # Sampling Functions
# # These functions sample a subset of listings and return
# # calendar and review rows that match those listings.
# # ---------------------------------------------------------

# def sample_listing_ids(df_listings, n_listings=10000):
#     """Randomly sample listing IDs from listings table."""
#     return df_listings["id"].sample(n_listings, random_state=42).unique()

# def sample_calendar(df_calendar, sampled_ids, start_date=None):
#     """Filter calendar rows to sampled listing IDs and optional date range."""
#     df = df_calendar[df_calendar["listing_id"].isin(sampled_ids)]
#     if start_date:
#         df = df[df["date"] >= pd.to_datetime(start_date)]
#     return df

# def sample_reviews(df_reviews, sampled_ids):
#     """Filter reviews to sampled listing IDs."""
#     return df_reviews[df_reviews["listing_id"].isin(sampled_ids)]

# # ---------------------------------------------------------
# # Sample for exploration
# # This gives you a consistent subset of listings, calendar,
# # and reviews — all tied to the same listing IDs.
# # ---------------------------------------------------------

# # Sample listing IDs
# sampled_ids = sample_listing_ids(df_listings, n_listings=10000)

# # Sample calendar and reviews based on those IDs
# df_calendar = sample_calendar(df_calendar, sampled_ids)
# df_reviews = sample_reviews(df_reviews, sampled_ids)

# # Filter listings to match sampled IDs
# df_listings = df_listings[df_listings["id"].isin(sampled_ids)]

🥇 Gold Table: listing_features
🎯 Goal
Create one row per listing with:
- Property attributes (from listings_clean)
- Review stats (from reviews_clean)
- Occupancy and pricing metrics (from calendar_clean)

In [5]:
# ---------------------------------------------------------
# GOLD STEP: Aggregate Calendar Metrics
# Purpose: Calculate occupancy rate and average price per listing
# from calendar data.
# ---------------------------------------------------------

calendar_agg = (
    df_calendar
    .groupby("listing_id")
    .agg(
        calendar_days=("date", "count"),
        available_days=("available", "sum"),
        avg_price=("price", "mean")
    )
    .assign(
        occupancy_rate=lambda df: 1 - (df["available_days"] / df["calendar_days"])
    )
    .reset_index()
)

print(f"✅ Calendar metrics aggregated for {len(calendar_agg):,} listings")

✅ Calendar metrics aggregated for 94,554 listings


In [6]:
# ---------------------------------------------------------
# GOLD STEP: Aggregate Review Metrics
# Purpose: Calculate review count and first/last review dates
# per listing.
# ---------------------------------------------------------

review_agg = (
    df_reviews
    .groupby("listing_id")
    .agg(
        review_count=("id", "count"),
        first_review_date=("date", "min"),
        last_review_date=("date", "max")
    )
    .reset_index()
)

print(f"✅ Review metrics aggregated for {len(review_agg):,} listings")

✅ Review metrics aggregated for 70,316 listings


In [7]:
# ---------------------------------------------------------
# GOLD STEP: Combine Listing, Calendar, and Review Features
# Purpose: Create a unified listing_features table with
# attributes and metrics from all sources.
# ---------------------------------------------------------

listing_features = (
    df_listings
    .merge(calendar_agg, how="left", left_on="id", right_on="listing_id")
    .merge(review_agg, how="left", on="listing_id")
)

# Optional: drop duplicate listing_id column
listing_features = listing_features.drop(columns=["listing_id"])

# Create a 1/0 flag: 1 if 't', else 0 (NaNs become False → 0)
listing_features["superhost_flag"] = listing_features["host_is_superhost"].eq("t").astype(int)

print(f"✅ Final listing_features table: {len(listing_features):,} rows")

✅ Final listing_features table: 94,559 rows


In [8]:
# ---------------------------------------------------------
# GOLD STEP: Save listing_features to disk
# Purpose: Store the final Gold table for dashboards, ML, or
# portfolio use.
# ---------------------------------------------------------

from pathlib import Path

# Resolve project root (assumes notebook is in /notebooks)
notebook_dir = Path().resolve()
project_root = notebook_dir.parent
gold_dir = project_root / "data" / "gold"
gold_dir.mkdir(parents=True, exist_ok=True)

# Save to Gold
output_path = gold_dir / "listing_features.parquet"
listing_features.to_parquet(output_path, index=False)

print(f"✅ Saved listing_features to {output_path}")

✅ Saved listing_features to C:\Users\emand\Documents\Python\DSPP  - AirBnBProject\data\gold\listing_features.parquet


📦 Final Schema (example)
Your listing_features table will include:
- id, room_type, beds, neighbourhood_cleansed, host_is_superhost, etc.
- calendar_days, available_days, avg_price, occupancy_rate
- review_count, first_review_date, last_review_date


In [9]:
# ---------------------------------------------------------
# GOLD STEP: Save neighbourhood_summary to disk
# Purpose: Aggregate listing features by neighbourhood for
# dashboards, ML, or portfolio use.
# ---------------------------------------------------------

from pathlib import Path

# Resolve project root (assumes notebook is in /notebooks)
notebook_dir = Path().resolve()
project_root = notebook_dir.parent
gold_dir = project_root / "data" / "gold"
gold_dir.mkdir(parents=True, exist_ok=True)

# Load listing_features
df_lf = listing_features

# Aggregate by neighbourhood
neighbourhood_summary = (
    df_lf
    .groupby("neighbourhood_cleansed")
    .agg(
        total_listings        = ("id",            "nunique"),
        avg_price             = ("avg_price",     "mean"),
        median_price          = ("avg_price",     "median"),
        avg_occupancy_rate    = ("occupancy_rate","mean"),
        avg_review_count      = ("review_count",  "mean"),
        superhost_share       = ("superhost_flag","mean")
    )
    .reset_index()
)

print(f"✅ Neighbourhood summary created: {len(neighbourhood_summary):,} rows")

# Save to Gold
output_path = gold_dir / "neighbourhood_summary.parquet"
neighbourhood_summary.to_parquet(output_path, index=False)

print(f"✅ Saved neighbourhood_summary to {output_path}")

✅ Neighbourhood summary created: 33 rows
✅ Saved neighbourhood_summary to C:\Users\emand\Documents\Python\DSPP  - AirBnBProject\data\gold\neighbourhood_summary.parquet


C:\Users\emand\AppData\Local\Temp\ipykernel_25832\37842345.py:21: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("neighbourhood_cleansed")


In [10]:
# load just enough to inspect
df_lf = listing_features
print("Rows:", len(df_lf))
print("Unique host_id:", df_lf["host_id"].nunique())
print("Unique host_name:", df_lf["host_name"].nunique())

Rows: 94559
Unique host_id: 55395
Unique host_name: 16416


In [11]:
# ---------------------------------------------------------
# GOLD STEP: Host Performance (optimized)
# Purpose: Group by host_id (fast) then attach host_name.
# ---------------------------------------------------------

import pandas as pd
from pathlib import Path

# Resolve project root (assumes notebook is in /notebooks)
notebook_dir = Path().resolve()
project_root = notebook_dir.parent
gold_dir = project_root / "data" / "gold"
gold_dir.mkdir(parents=True, exist_ok=True)

# Load only the cols we need for aggregation
cols = [
    "id", "host_id", "host_name",
    "avg_price", "occupancy_rate",
    "review_count", "superhost_flag"
]
df = pd.read_parquet(gold_dir / "listing_features.parquet", columns=cols)

# 1) Extract host_name lookup (one row per host)
host_lookup = (
    df[["host_id", "host_name"]]
    .drop_duplicates(subset="host_id")
    .set_index("host_id")
)

# 2) Aggregate by host_id alone
host_perf = (
    df
    .groupby("host_id", sort=False)
    .agg(
        listing_count        = ("id",              "nunique"),
        avg_price            = ("avg_price",       "mean"),
        avg_occupancy_rate   = ("occupancy_rate",  "mean"),
        total_review_count   = ("review_count",    "sum"),
        superhost_rate       = ("superhost_flag",  "mean")
    )
)

# 3) Re-attach host_name
host_perf = host_perf.join(host_lookup, how="left").reset_index()

# 4) Persist
output_path = gold_dir / "host_performance.parquet"
host_perf.to_parquet(output_path, index=False)

print(f"✅ {len(host_perf):,} hosts saved to {output_path.name}")

✅ 55,395 hosts saved to host_performance.parquet


In [12]:
# ---------------------------------------------------------
# GOLD STEP: Time Series Trends
# Purpose: Compute monthly market dynamics for line charts.
# ---------------------------------------------------------

import pandas as pd
from pathlib import Path

# Resolve project root (assumes notebook is in /notebooks)
notebook_dir = Path().resolve()
project_root = notebook_dir.parent
gold_dir = project_root / "data" / "gold"
gold_dir.mkdir(parents=True, exist_ok=True)

# Create year_month period
df_calendar["year_month"] = df_calendar["date"].dt.to_period("M").astype(str)
df_reviews["year_month"] = df_reviews["date"].dt.to_period("M").astype(str)

# Monthly calendar aggregates
calendar_ts = (
    df_calendar
    .groupby("year_month")
    .agg(
        listings_live=("listing_id", "nunique"),
        avg_price=("price", "mean"),
        avg_occupancy_rate=("available", lambda x: 1 - x.sum() / x.count())
    )
    .reset_index()
)

# Monthly review counts
reviews_ts = (
    df_reviews
    .groupby("year_month")
    .agg(review_count=("id", "count"))
    .reset_index()
)

# New listings per month (first calendar date)
first_calendar = (
    df_calendar
    .groupby("listing_id")["date"]
    .min()
    .dt.to_period("M")
    .astype(str)
    .reset_index(name="first_month")
)
new_listings = (
    first_calendar
    .groupby("first_month")
    .size()
    .reset_index(name="new_listings")
    .rename(columns={"first_month": "year_month"})
)

# Combine all time series
time_series = (
    calendar_ts
    .merge(reviews_ts, on="year_month", how="left")
    .merge(new_listings, on="year_month", how="left")
)

print(f"✅ Time series table created: {len(time_series):,} months")

# Save to Gold
output_path = gold_dir / "time_series.parquet"
time_series.to_parquet(output_path, index=False)

print(f"✅ Saved time_series to {output_path}")

✅ Time series table created: 13 months
✅ Saved time_series to C:\Users\emand\Documents\Python\DSPP  - AirBnBProject\data\gold\time_series.parquet


In [13]:
# ---------------------------------------------------------
# GOLD STEP: Room Type Breakdown
# Purpose: Compare performance across room types for bar
# charts and slicers.
# ---------------------------------------------------------

import pandas as pd
from pathlib import Path

# Resolve project root (assumes notebook is in /notebooks)
notebook_dir = Path().resolve()
project_root = notebook_dir.parent
gold_dir = project_root / "data" / "gold"
gold_dir.mkdir(parents=True, exist_ok=True)

# Aggregate by room type
room_type_breakdown = (
    df_lf
    .groupby("room_type")
    .agg(
        listing_count=("id", "nunique"),
        avg_price=("avg_price", "mean"),
        avg_occupancy_rate=("occupancy_rate", "mean"),
        avg_review_count=("review_count", "mean")
    )
    .reset_index()
)

print(f"✅ Room type breakdown created: {len(room_type_breakdown):,} categories")

# Save to Gold
output_path = gold_dir / "room_type_breakdown.parquet"
room_type_breakdown.to_parquet(output_path, index=False)

print(f"✅ Saved room_type_breakdown to {output_path}")

✅ Room type breakdown created: 4 categories
✅ Saved room_type_breakdown to C:\Users\emand\Documents\Python\DSPP  - AirBnBProject\data\gold\room_type_breakdown.parquet


C:\Users\emand\AppData\Local\Temp\ipykernel_25832\1041901986.py:19: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("room_type")
